# 00 – Chuẩn bị môi trường trên Colab

Chạy notebook này **một lần** cho mỗi tài khoản Colab. Dữ liệu đã xử lý, mô hình VnCoreNLP và kết quả
được lưu trên Google Drive, các notebook mô hình sau chỉ cần chạy 3 ô đầu (GPU, Drive, clone repo).

In [ ]:
!nvidia-smi -L || echo 'Không có GPU – vào Runtime > Change runtime type > T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'   # thư mục chung của nhóm trên Drive

In [ ]:
REPO_URL = 'https://github.com/quanuiter/NLI.git'
!git clone -q $REPO_URL /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt

Trỏ `data/processed`, `tools` (VnCoreNLP) và `results` sang Drive để không mất khi Colab ngắt.
Nhớ tải `results/` về và commit lên GitHub sau mỗi đợt thí nghiệm.

In [ ]:
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub)
    os.makedirs(target, exist_ok=True)
    if os.path.islink(sub):
        continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.makedirs(os.path.dirname(sub) or '.', exist_ok=True)
    os.symlink(target, sub)
!ls -la data results tools

In [ ]:
!python scripts/prepare_data.py

Tách từ cho PhoBERT (VnCoreNLP cần Java, Colab có sẵn). Chạy một lần, khoảng vài chục phút cho ~200k câu.

In [ ]:
!python scripts/segment_data.py --check 20

In [ ]:
!python tests/test_core.py

In [ ]:
from src.data import load_scenario
for s in ['S1', 'S2', 'S3']:
    d = load_scenario(s)
    print(s, {k: len(v) if not isinstance(v, dict) else {t: len(x) for t, x in v.items()} for k, v in d.items()})
load_scenario('S2', segmented=True)['train'].head()